In [3]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack")

# Select only these folders
SELECTED_FOLDERS = {
    "collections": "merged_collections.csv",
    "patients": "merged_patient.csv",
    "invoices": "merged_invoices.csv"
}

OUTPUT_DIR = BASE_DIR / "merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for folder_name, output_name in SELECTED_FOLDERS.items():

    folder_path = BASE_DIR / folder_name

    if not folder_path.is_dir():
        print(f"Folder not found: {folder_path}")
        continue

    gz_files = sorted(folder_path.rglob("*.gz"))

    if not gz_files:
        print(f"No .gz files found in {folder_name}")
        continue

    output_file = OUTPUT_DIR / output_name
    first_chunk = True
    total_rows = 0

    print(f"\nMerging folder: {folder_name}")

    for file in gz_files:
        print(f"Reading: {file.name}")

        for chunk in pd.read_csv(
            file,
            compression="gzip",
            chunksize=100_000
        ):
            chunk.to_csv(
                output_file,
                mode="w" if first_chunk else "a",
                index=False,
                header=first_chunk
            )

            first_chunk = False
            total_rows += len(chunk)

    print(f"Files processed: {len(gz_files)}")
    print(f"Rows merged: {total_rows:,}")
    print(f"Saved as: {output_file}")

print("\nAll selected folders merged successfully!")



Merging folder: collections
Reading: collections_AHM_2025-01.csv.gz
Reading: collections_AHM_2025-02.csv.gz
Reading: collections_AHM_2025-03.csv.gz
Reading: collections_AHM_2025-04.csv.gz
Reading: collections_AHM_2025-05.csv.gz
Reading: collections_AHM_2025-06.csv.gz
Reading: collections_AHM_2025-07.csv.gz
Reading: collections_AHM_2025-08.csv.gz
Reading: collections_AHM_2025-09.csv.gz
Reading: collections_AHM_2025-10.csv.gz
Reading: collections_AHM_2025-11.csv.gz
Reading: collections_AHM_2025-12.csv.gz
Reading: collections_AHM_2026-01.csv.gz
Reading: collections_AHM_2026-02.csv.gz
Reading: collections_AHM_2026-03.csv.gz
Reading: collections_AHM_2026-04.csv.gz
Reading: collections_AHM_2026-05.csv.gz
Reading: collections_AHM_2026-06.csv.gz
Reading: collections_BLR_2025-01.csv.gz
Reading: collections_BLR_2025-02.csv.gz
Reading: collections_BLR_2025-03.csv.gz
Reading: collections_BLR_2025-04.csv.gz
Reading: collections_BLR_2025-05.csv.gz
Reading: collections_BLR_2025-06.csv.gz
Reading: co

In [4]:
CAMPAIGN_BASE_DIR = Path(
    r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\campaigns"
)
OUTPUT_DIR = BASE_DIR / "merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


gz_files=list(CAMPAIGN_BASE_DIR.glob("*.gz"))

all_dataframes=[]
for file in gz_files:
    df=pd.read_csv(file)
    all_dataframes.append(df)

combined_df=pd.concat(all_dataframes,ignore_index=True)

output_file=OUTPUT_DIR/"combined_campaigns.csv"
combined_df.to_csv(output_file,index=False)

print(f"Successfully combined {len(gz_files)} files into: {output_file}")

Successfully combined 1 files into: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\merged\combined_campaigns.csv


In [5]:
merged_campaigns=pd.read_csv(OUTPUT_DIR/"combined_campaigns.csv")

In [6]:
merged_campaigns.head()

,booking_id,campaign_id,waiver_rs
0,BK-AHM-202601-065286,HC-FREE-2026Q1,150
1,BK-AHM-202601-065292,HC-FREE-2026Q1,150
2,BK-AHM-202601-065298,HC-FREE-2026Q1,150
3,BK-AHM-202601-065301,HC-FREE-2026Q1,150
4,BK-AHM-202601-065306,HC-FREE-2026Q1,150


In [7]:
DIR = Path(r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\merged")

collections=pd.read_csv(DIR/"merged_collections.csv")
invoices=pd.read_csv(DIR/"merged_invoices.csv")
patient=pd.read_csv(DIR/"merged_patient.csv")

In [10]:

from pathlib import Path
import pandas as pd

# Main project directory
BASE_DIR = Path(
    r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack"
)

# Output directory
OUTPUT_DIR = BASE_DIR / "merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def merge_gz_files(folder_path, output_file, add_columns):
    """Merge all gzip-compressed CSV files into one CSV."""

    gz_files = sorted(folder_path.rglob("*.gz"))

    if not gz_files:
        print(f"No .gz files found in {folder_path}")
        return

    # Remove an old output file before rebuilding it
    if output_file.exists():
        output_file.unlink()

    first_chunk = True
    total_rows = 0
    successful_files = 0

    for file in gz_files:
        print(f"Reading: {file}")

        extra_columns = add_columns(file)

        try:
            for chunk in pd.read_csv(
                file,
                compression="gzip",
                encoding="latin1",
                chunksize=100_000
            ):
                # Add metadata columns
                for column, value in extra_columns.items():
                    chunk[column] = value

                chunk.to_csv(
                    output_file,
                    mode="w" if first_chunk else "a",
                    index=False,
                    header=first_chunk
                )

                first_chunk = False
                total_rows += len(chunk)

            successful_files += 1

        except Exception as e:
            print(f"ERROR processing {file}: {e}")

    print(f"\nOutput: {output_file}")
    print(f"Files found: {len(gz_files)}")
    print(f"Files processed successfully: {successful_files}")
    print(f"Total rows written: {total_rows:,}")


# --------------------------------------------------
# 1. RESULTS: add city column
# --------------------------------------------------

RESULTS_DIR = BASE_DIR / "results"


def results_metadata(file):
    # Structure: results/city_name/file.gz
    city = file.relative_to(RESULTS_DIR).parts[0]
    return {"city": city}


merge_gz_files(
    folder_path=RESULTS_DIR,
    output_file=OUTPUT_DIR / "merged_results.csv",
    add_columns=results_metadata
)


# --------------------------------------------------
# 2. BOOKINGS: add booking type and city columns
# --------------------------------------------------

BOOKINGS_DIR = BASE_DIR / "bookings"


def bookings_metadata(file):
    # Structure: bookings/legacy/city_name/file.gz
    #            bookings/nova/city_name/file.gz
    relative_parts = file.relative_to(BOOKINGS_DIR).parts

    booking_type = relative_parts[0]
    city = relative_parts[1]

    return {
        "type_of_booking": booking_type,
        "city": city
    }


merge_gz_files(
    folder_path=BOOKINGS_DIR,
    output_file=OUTPUT_DIR / "merged_bookings.csv",
    add_columns=bookings_metadata
)

print("\nProcessing completed!")


Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-01.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-02.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-03.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-04.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-05.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-06.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-07.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-08.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\results_2025-09.csv.gz
Reading: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\results\ahmedabad\res

In [1]:
import glob
import gzip
import json
import pandas as pd
from pathlib import Path

# Main data folder
BASE_DIR = Path(
    r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack"
)

# Find all compressed JSONL booking files under Nova
files = glob.glob(
    str(BASE_DIR / "bookings" / "nova" / "**" / "*.jsonl.gz"),
    recursive=True
)

all_records = []

for file in files:
    with gzip.open(file, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                all_records.append(json.loads(line))

# Flatten nested JSON dictionaries
bookings = pd.json_normalize(all_records, sep="_")

# Extract test details
bookings["test_codes"] = bookings["tests"].apply(
    lambda tests: ", ".join(str(t["code"]) for t in tests)
)

bookings["test_prices"] = bookings["tests"].apply(
    lambda tests: ", ".join(str(t["price"]) for t in tests)
)

bookings["test_count"] = bookings["tests"].apply(len)

# Drop nested list column
bookings = bookings.drop(columns=["tests"])

# Save merged CSV
output_file = BASE_DIR / "merged" / "nova_bookings.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)

bookings.to_csv(output_file, index=False)

print("Files found:", len(files))
print("Total bookings:", len(bookings))
print("Saved to:", output_file)


Files found: 6
Total bookings: 26543
Saved to: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\merged\nova_bookings.csv


In [11]:
nova=pd.read_csv(DIR/"nova_bookings.csv")

In [12]:
nova.head()

,booking_id,created_at,slot_at,city,clinic_id,channel,source,home_fee,rescheduled_from,status_at_export,patient_patient_id,patient_is_new,test_codes,test_prices,test_count
0,nova_che_0087556,2026-04-21T01:57:00+05:30,2026-04-25T10:16:00+05:30,Chennai,CHE-01,CLINIC,WALKIN,0,NaN,booked,PCHE0044922,False,T007,550,1
1,nova_che_0087557,2026-03-28T03:02:00+05:30,2026-04-01T09:17:00+05:30,Chennai,CHE-03,CLINIC,PHONE,0,NaN,booked,PCHE0047035,False,"T007, T011, T004","550, 750, 650",3
2,nova_che_0087558,2026-04-28T04:46:00+05:30,2026-04-30T08:19:00+05:30,Chennai,CHE-05,CLINIC,PHONE,0,NaN,booked,PCHE0051652,False,"T004, T001","650, 350",2
3,nova_che_0087559,2026-04-16T05:53:00+05:30,2026-04-21T08:08:00+05:30,Chennai,CHE-02,HOME,APP,150,NaN,booked,PCHE0040733,False,"T006, T007, T017, T008, PK03","650, 550, 450, 1100, 999",5
4,nova_che_0087560,2026-04-07T05:57:00+05:30,2026-04-08T08:46:00+05:30,Chennai,CHE-05,CLINIC,WALKIN,0,NaN,rescheduled,PCHE0037380,False,T007,550,1


In [17]:
nova.dtypes

booking_id              str
created_at              str
slot_at                 str
city                    str
clinic_id               str
channel                 str
source                  str
home_fee              int64
rescheduled_from        str
status_at_export        str
patient_patient_id      str
patient_is_new         bool
test_codes              str
test_prices             str
test_count            int64
dtype: object

In [20]:
nova.size

398145

In [13]:
bookings=pd.read_csv(DIR/"merged_bookings.csv")
results=pd.read_csv(DIR/"merged_results.csv")

C:\Users\Dharmen\AppData\Local\Temp\ipykernel_12068\2525229462.py:1: DtypeWarning: Columns (10: test_price, 11: home_fee) have mixed types. Specify dtype option on import or set low_memory=False.
  bookings=pd.read_csv(DIR/"merged_bookings.csv")


In [32]:
bookings.dtypes

booking_id            str
created_at            str
slot_at               str
city                  str
clinic_id             str
channel               str
source                str
patient_id            str
patient_name          str
test_code             str
test_price         object
home_fee           object
type_of_booking       str
dtype: object

In [23]:
nova.dtypes

booking_id              str
created_at              str
slot_at                 str
city                    str
clinic_id               str
channel                 str
source                  str
home_fee              int64
rescheduled_from        str
status_at_export        str
patient_patient_id      str
patient_is_new         bool
test_codes              str
test_prices             str
test_count            int64
dtype: object

In [22]:
bookings.dtypes

booking_id            str
created_at            str
slot_at               str
city                  str
clinic_id             str
channel               str
source                str
patient_id            str
patient_name          str
test_code             str
test_price         object
home_fee           object
type_of_booking       str
dtype: object

In [14]:
bookings.head()

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,type_of_booking
0,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,legacy
1,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,legacy
2,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,legacy
3,BK-AHM-202501-000002,03/01/2025 05:18,06/01/2025 13:22,ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,legacy
4,BK-AHM-202501-000003,27/01/2025 10:46,28/01/2025 13:37,ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,legacy


In [47]:

import pandas as pd

# Work on copies so the original DataFrames remain unchanged
legacy = bookings.copy()
new = nova.copy()

# 1. Standardize Nova column names
new = new.rename(columns={
    "patient_patient_id": "patient_id"
})

# 2. Convert comma-separated test codes/prices into lists
def to_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or value == "":
        return []
    return [x.strip() for x in str(value).split(",")]

new["test_codes"] = new["test_codes"].apply(to_list)
new["test_prices"] = new["test_prices"].apply(to_list)

# Verify that every test code has a corresponding price
mismatch = new.apply(
    lambda r: len(r["test_codes"]) != len(r["test_prices"]),
    axis=1
)

if mismatch.any():
    raise ValueError(
        "Some Nova bookings have mismatched test codes and prices:\n"
        + str(new.loc[mismatch, ["booking_id", "test_codes", "test_prices"]])
    )

# 3. Expand Nova to one row per test
new["test_pair"] = new.apply(
    lambda r: list(zip(r["test_codes"], r["test_prices"])),
    axis=1
)

new = new.explode("test_pair", ignore_index=True)

new["test_code"] = new["test_pair"].apply(
    lambda x: x[0] if isinstance(x, tuple) else pd.NA
)
new["test_price"] = new["test_pair"].apply(
    lambda x: x[1] if isinstance(x, tuple) else pd.NA
)

new = new.drop(
    columns=["test_codes", "test_prices", "test_count", "test_pair"],
    errors="ignore"
)

# 4. Map Nova's booking type
# Confirm that CLINIC/HOME represents booking type in your data.
new["type_of_booking"] = new["channel"]

# 5. Add fields missing from either source
for col in ["patient_name", "patient_is_new",
            "rescheduled_from", "status_at_export"]:
    if col not in new.columns:
        new[col] = pd.NA
    if col not in legacy.columns:
        legacy[col] = pd.NA

# 6. Add a source identifier
legacy["data_source"] = "LEGACY"
new["data_source"] = "NOVA"

# 7. Align column names and order
common_columns = [
    "booking_id",
    "created_at",
    "slot_at",
    "city",
    "clinic_id",
    "channel",
    "source",
    "patient_id",
    "patient_name",
    "test_code",
    "test_price",
    "home_fee",
    "type_of_booking",
    "rescheduled_from",
    "patient_is_new",
    "status_at_export",
    "data_source"
]

for col in common_columns:
    if col not in legacy.columns:
        legacy[col] = pd.NA
    if col not in new.columns:
        new[col] = pd.NA

legacy = legacy[common_columns].copy()
new = new[common_columns].copy()

# 8. Combine the tables
combined = pd.concat(
    [legacy, new],
    ignore_index=True
)

# 9. Standardize data types
for col in ["created_at", "slot_at"]:
    combined[col] = pd.to_datetime(
        combined[col], errors="coerce", utc=True
    )

for col in ["test_price", "home_fee"]:
    combined[col] = pd.to_numeric(
        combined[col], errors="coerce"
    )

# 10. Save the uniform dataset
output_path = (
    r"C:\Users\Dharmen\Downloads\Build1_DataPack"
    r"\datapack\merged\uniform_bookings_v2.csv"
)

from pathlib import Path
Path(output_path).parent.mkdir(parents=True, exist_ok=True)

combined.to_csv(output_path, index=False)

print("Legacy rows:", len(legacy))
print("Nova rows after test expansion:", len(new))
print("Combined rows:", len(combined))
print("Unique bookings:", combined["booking_id"].nunique())
print("Saved to:", output_path)


Legacy rows: 1252786
Nova rows after test expansion: 62439
Combined rows: 1315225
Unique bookings: 568008
Saved to: C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\merged\uniform_bookings_v2.csv


In [39]:
# Parse Nova dates, preserving their timezone
new["created_at"] = pd.to_datetime(
    new["created_at"], format="ISO8601", errors="coerce", utc=True
)

new["slot_at"] = pd.to_datetime(
    new["slot_at"], format="ISO8601", errors="coerce", utc=True
)

# Parse Legacy dates using day/month/year format
legacy["created_at"] = pd.to_datetime(
    legacy["created_at"], format="%d/%m/%Y %H:%M",
    errors="coerce"
)

legacy["slot_at"] = pd.to_datetime(
    legacy["slot_at"], format="%d/%m/%Y %H:%M",
    errors="coerce"
)

In [40]:
legacy["created_at"] = legacy["created_at"].dt.tz_localize(
    "Asia/Kolkata"
).dt.tz_convert("UTC")

legacy["slot_at"] = legacy["slot_at"].dt.tz_localize(
    "Asia/Kolkata"
).dt.tz_convert("UTC")

In [49]:
final_bookings=pd.read_csv(r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\merged\uniform_bookings_v2.csv")

C:\Users\Dharmen\AppData\Local\Temp\ipykernel_12068\835186346.py:1: DtypeWarning: Columns (8: patient_name, 13: rescheduled_from, 14: patient_is_new, 15: status_at_export) have mixed types. Specify dtype option on import or set low_memory=False.
  final_bookings=pd.read_csv(r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\merged\uniform_bookings_v2.csv")


In [50]:
final_bookings

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,type_of_booking,rescheduled_from,patient_is_new,status_at_export,data_source
0,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800.0,0.0,legacy,NaN,NaN,NaN,LEGACY
1,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650.0,0.0,legacy,NaN,NaN,NaN,LEGACY
2,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550.0,0.0,legacy,NaN,NaN,NaN,LEGACY
3,BK-AHM-202501-000002,2025-01-02 23:48:00+00:00,2025-01-06 07:52:00+00:00,ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650.0,0.0,legacy,NaN,NaN,NaN,LEGACY
4,BK-AHM-202501-000003,2025-01-27 05:16:00+00:00,2025-01-28 08:07:00+00:00,ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450.0,150.0,legacy,NaN,NaN,NaN,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1315220,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,HOME,PHONE,PJAI0017296,NaN,T007,550.0,150.0,HOME,nova_jai_0052790,False,booked,NOVA
1315221,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,HOME,PHONE,PJAI0017296,NaN,T006,650.0,150.0,HOME,nova_jai_0052790,False,booked,NOVA
1315222,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,CLINIC,PHONE,PJAI0027566,NaN,T006,650.0,0.0,CLINIC,nova_jai_0054062,False,booked,NOVA
1315223,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,CLINIC,PHONE,PJAI0027566,NaN,T002,120.0,0.0,CLINIC,nova_jai_0054062,False,booked,NOVA


In [51]:

lst=['channel','source','city']
for i in lst:
    final_bookings[i]=final_bookings[i].str.capitalize()
    

In [52]:
final_bookings

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,type_of_booking,rescheduled_from,patient_is_new,status_at_export,data_source
0,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T009,800.0,0.0,legacy,NaN,NaN,NaN,LEGACY
1,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T004,650.0,0.0,legacy,NaN,NaN,NaN,LEGACY
2,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T007,550.0,0.0,legacy,NaN,NaN,NaN,LEGACY
3,BK-AHM-202501-000002,2025-01-02 23:48:00+00:00,2025-01-06 07:52:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0031384,Nisha Iyer,T004,650.0,0.0,legacy,NaN,NaN,NaN,LEGACY
4,BK-AHM-202501-000003,2025-01-27 05:16:00+00:00,2025-01-28 08:07:00+00:00,Ahmedabad,AHM-01,Home,App,PAHM0027001,Sneha Menon,T003,450.0,150.0,legacy,NaN,NaN,NaN,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1315220,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,T007,550.0,150.0,HOME,nova_jai_0052790,False,booked,NOVA
1315221,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,T006,650.0,150.0,HOME,nova_jai_0052790,False,booked,NOVA
1315222,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,Clinic,Phone,PJAI0027566,NaN,T006,650.0,0.0,CLINIC,nova_jai_0054062,False,booked,NOVA
1315223,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,Clinic,Phone,PJAI0027566,NaN,T002,120.0,0.0,CLINIC,nova_jai_0054062,False,booked,NOVA


In [53]:
final_copy=final_bookings.copy()

In [67]:
final_bookings=final_bookings.drop(columns=['status_at_export','patient_is_new','rescheduled_from','type_of_booking'])

In [68]:
final_bookings['test_code'].isna().sum()

np.int64(1311)

In [ ]:
final_bookings

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,data_source
0,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T009,800.0,0.0,LEGACY
1,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T004,650.0,0.0,LEGACY
2,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T007,550.0,0.0,LEGACY
3,BK-AHM-202501-000002,2025-01-02 23:48:00+00:00,2025-01-06 07:52:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0031384,Nisha Iyer,T004,650.0,0.0,LEGACY
4,BK-AHM-202501-000003,2025-01-27 05:16:00+00:00,2025-01-28 08:07:00+00:00,Ahmedabad,AHM-01,Home,App,PAHM0027001,Sneha Menon,T003,450.0,150.0,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1315220,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,T007,550.0,150.0,NOVA
1315221,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,T006,650.0,150.0,NOVA
1315222,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,Clinic,Phone,PJAI0027566,NaN,T006,650.0,0.0,NOVA
1315223,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,Clinic,Phone,PJAI0027566,NaN,T002,120.0,0.0,NOVA


In [80]:
final_bookings['created_at']=pd.to_datetime(final_bookings['created_at'])
final_bookings['slot_at']=pd.to_datetime(final_bookings['slot_at'])

In [95]:
lolo=(final_bookings['booking_id']=='booking_id')
final_bookings = final_bookings[~lolo]

In [96]:
final_bookings

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,data_source
0,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T009,800.0,0.0,LEGACY
1,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T004,650.0,0.0,LEGACY
2,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T007,550.0,0.0,LEGACY
3,BK-AHM-202501-000002,2025-01-02 23:48:00+00:00,2025-01-06 07:52:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0031384,Nisha Iyer,T004,650.0,0.0,LEGACY
4,BK-AHM-202501-000003,2025-01-27 05:16:00+00:00,2025-01-28 08:07:00+00:00,Ahmedabad,AHM-01,Home,App,PAHM0027001,Sneha Menon,T003,450.0,150.0,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1315220,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,T007,550.0,150.0,NOVA
1315221,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,T006,650.0,150.0,NOVA
1315222,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,Clinic,Phone,PJAI0027566,NaN,T006,650.0,0.0,NOVA
1315223,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,Clinic,Phone,PJAI0027566,NaN,T002,120.0,0.0,NOVA


In [104]:
output_path = r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\merged\formatted_uniform_bookings.csv"

final_bookings.to_csv(output_path, index=False)

Booking for current and last half year

In [97]:
cond=(final_bookings['created_at']>='2025-01-01')&(final_bookings['created_at']<'2025-07-01')
new_cond=(final_bookings['created_at']>='2026-01-01')&(final_bookings['created_at']<'2026-07-01')

In [98]:
last_year_bookings=final_bookings[cond]
last_year_bookings

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,data_source
0,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T009,800.0,0.0,LEGACY
1,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T004,650.0,0.0,LEGACY
2,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,T007,550.0,0.0,LEGACY
3,BK-AHM-202501-000002,2025-01-02 23:48:00+00:00,2025-01-06 07:52:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0031384,Nisha Iyer,T004,650.0,0.0,LEGACY
4,BK-AHM-202501-000003,2025-01-27 05:16:00+00:00,2025-01-28 08:07:00+00:00,Ahmedabad,AHM-01,Home,App,PAHM0027001,Sneha Menon,T003,450.0,150.0,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1138191,BK-PUN-202507-030682,2025-06-29 03:25:00+00:00,2025-07-03 04:55:00+00:00,Pune,PUN-02,Home,Phone,PPUN0032522,Ritika Shah,T005,700.0,150.0,LEGACY
1138218,BK-PUN-202507-030696,2025-06-27 22:31:00+00:00,2025-07-01 07:25:00+00:00,Pune,PUN-03,Clinic,Walkin,PPUN0032536,Saanvi Iyer,T001,350.0,0.0,LEGACY
1138219,BK-PUN-202507-030696,2025-06-27 22:31:00+00:00,2025-07-01 07:25:00+00:00,Pune,PUN-03,Clinic,Walkin,PPUN0032536,Saanvi Iyer,PK01,1499.0,0.0,LEGACY
1138220,BK-PUN-202507-030696,2025-06-27 22:31:00+00:00,2025-07-01 07:25:00+00:00,Pune,PUN-03,Clinic,Walkin,PPUN0032536,Saanvi Iyer,T002,120.0,0.0,LEGACY


In [99]:
present_year_bookings=final_bookings[new_cond]
present_year_bookings

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,data_source
150289,BK-AHM-202601-065284,2026-01-17 23:47:00+00:00,2026-01-23 05:24:00+00:00,Ahmedabad,AHM-03,Clinic,App,PAHM0008918,Varun Bose,T008,1100.0,0.0,LEGACY
150290,BK-AHM-202601-065284,2026-01-17 23:47:00+00:00,2026-01-23 05:24:00+00:00,Ahmedabad,AHM-03,Clinic,App,PAHM0008918,Varun Bose,T018,700.0,0.0,LEGACY
150291,BK-AHM-202601-065285,2026-01-09 00:42:00+00:00,2026-01-13 04:21:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0042707,Farhan Singh,T012,600.0,0.0,LEGACY
150292,BK-AHM-202601-065285,2026-01-09 00:42:00+00:00,2026-01-13 04:21:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0042707,Farhan Singh,T003,450.0,0.0,LEGACY
150293,BK-AHM-202601-065286,2026-01-12 20:40:00+00:00,2026-01-13 01:34:00+00:00,Ahmedabad,AHM-02,Home,App,PAHM0014637,Ishaan Desai,T011,750.0,0.0,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1315220,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,T007,550.0,150.0,NOVA
1315221,nova_jai_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,T006,650.0,150.0,NOVA
1315222,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,Clinic,Phone,PJAI0027566,NaN,T006,650.0,0.0,NOVA
1315223,nova_jai_0055753,2026-06-05 02:21:00+00:00,2026-06-06 04:55:00+00:00,Jaipur,JAI-03,Clinic,Phone,PJAI0027566,NaN,T002,120.0,0.0,NOVA


Test per visit over two years definite interval of 6 months

In [100]:
last_year_bookings['test_code'].count()/last_year_bookings['booking_id'].nunique()

np.float64(2.296593788021637)

In [101]:
present_year_bookings['test_code'].count()/present_year_bookings['booking_id'].nunique()

np.float64(2.3459216999462567)

In [102]:
last_year_bookings['patient_id'].nunique()

94624

In [103]:
present_year_bookings['patient_id'].nunique()

97170

In [69]:
final_bookings.size

17097925

In [70]:
final_bookings[final_bookings['test_code'].isna()]

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,data_source
1023,BK-AHM-202501-000462,2025-01-12 18:56:00+00:00,2025-01-13 02:55:00+00:00,Ahmedabad,AHM-01,Clinic,Walkin,PAHM0021638,Manish Sharma,NaN,550.0,0.0,LEGACY
1256,BK-AHM-202501-000572,2025-01-14 01:07:00+00:00,2025-01-18 07:38:00+00:00,Ahmedabad,AHM-03,Clinic,Phone,PAHM0001702,Renée Pillai,NaN,1100.0,0.0,LEGACY
2242,BK-AHM-202501-001008,2025-01-20 22:50:00+00:00,2025-01-26 06:20:00+00:00,Ahmedabad,AHM-05,Home,App,PAHM0028768,Amit Fernandes,NaN,2999.0,150.0,LEGACY
4963,BK-AHM-202501-002213,2025-01-07 04:56:00+00:00,2025-01-09 12:02:00+00:00,Ahmedabad,AHM-03,Clinic,App,PAHM0020634,Priya Desai,NaN,120.0,0.0,LEGACY
5116,BK-AHM-202501-002281,2025-01-01 01:52:00+00:00,2025-01-03 02:24:00+00:00,Ahmedabad,AHM-03,Home,App,PAHM0007861,Farhan Rao,NaN,800.0,150.0,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296986,nova_jai_0048026,2026-04-21 00:20:00+00:00,2026-04-23 07:23:00+00:00,Jaipur,JAI-04,Home,Walkin,PJAI0025509,NaN,NaN,NaN,150.0,NOVA
1301436,nova_jai_0049911,2026-05-11 01:18:00+00:00,2026-05-11 08:23:00+00:00,Jaipur,JAI-01,Clinic,App,PJAI0028443,NaN,NaN,NaN,0.0,NOVA
1307207,nova_jai_0052368,2026-05-06 17:42:00+00:00,2026-05-11 01:50:00+00:00,Jaipur,JAI-05,Clinic,App,PJAI0028776,NaN,NaN,NaN,0.0,NOVA
1309975,nova_jai_0053556,2026-06-27 23:38:00+00:00,2026-06-28 02:03:00+00:00,Jaipur,JAI-02,Clinic,App,PJAI0027145,NaN,NaN,NaN,0.0,NOVA


In [10]:
invoices.head()

,invoice_id,invoice_date,booking_id,patient_id,city,channel,amount
0,INV-2024-000001,2024-12-30,BK-BLR-202501-006763,PBLR0044005,Bengaluru,clinic,280.00
1,INV-2024-000002,2024-12-30,BK-BLR-202501-007103,PBLR0044345,Bengaluru,clinic,"1,070.00"
2,INV-2024-000003,2024-12-31,BK-PUN-202501-001985,PPUN0008860,Pune,clinic,350.00
3,INV-2024-000004,2024-12-31,BK-CHE-202501-001176,PCHE0012203,Chennai,home,"3,299.00"
4,INV-2025-000001,2025-01-01,BK-AHM-202501-005119,PAHM0034226,Ahmedabad,clinic,"2,649.00"


In [11]:
results.head()

,booking_id,test_code,result_ts,result_value,result_flag,city
0,BK-AHM-202501-000001,T009,2025-01-09 20:09:00,46.8,normal,ahmedabad
1,BK-AHM-202501-000001,T004,2025-01-10 00:09:00,49.0,normal,ahmedabad
2,BK-AHM-202501-000001,T007,2025-01-09 18:09:00,74.8,normal,ahmedabad
3,BK-AHM-202501-000002,T004,2025-01-07 15:44:00,52.7,flag,ahmedabad
4,BK-AHM-202501-000003,T003,2025-01-29 05:37:00,71.7,flag,ahmedabad


In [12]:
PAYMENT_BASE_DIR=Path(
    r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\payments"
)
MERGED_DIR=Path(r"C:\Users\Dharmen\Downloads\Build1_DataPack\datapack\merged")

MERGED_DIR.mkdir(parents=True, exist_ok=True)

def merge_files_by_pattern(pattern,file_output):

    matching_files=list(PAYMENT_BASE_DIR.glob(pattern))

    if not matching_files:
        print("Matching files not found")
        return
    print(f"Matching files for pattern: {pattern}")

    all_dfs=[]
    for file in matching_files:
        df=pd.read_csv(file)
        all_dfs.append(df)

    combined_df=pd.concat(all_dfs,ignore_index=True)

    output_path=MERGED_DIR/file_output
    combined_df.to_csv(output_path,index=False)

merge_files_by_pattern(pattern='gateway*.gz',file_output='merged_gateway.csv')

merge_files_by_pattern(pattern='settlements*.gz',file_output='merged_settlement.csv')


print("All processes complete!")
    

Matching files for pattern: gateway*.gz
Matching files for pattern: settlements*.gz
All processes complete!


In [13]:
gateway=pd.read_csv(MERGED_DIR/'merged_gateway.csv')
settlement=pd.read_csv(MERGED_DIR/'merged_settlement.csv')

In [14]:
gateway['created_date']=pd.to_datetime(gateway['created_at_epoch'],unit='s')
gateway.to_csv(MERGED_DIR / 'merged_gateway.csv', index=False)

In [15]:
gateway

,payment_id,notes_invoice_ref,amount_paise,status,created_at_epoch,method,settlement_id,created_date
0,pay_0000000002,INV-2024-000002,107000,captured,1735577378,upi,setl_67733400,2024-12-30 16:49:38
1,pay_0000000003,INV-2024-000003,35000,captured,1735653947,upi,setl_67748580,2024-12-31 14:05:47
2,pay_0000000001,INV-2024-000001,28000,captured,1735996468,upi,setl_6779cb80,2025-01-04 13:14:28
3,pay_0000000004,INV-2024-000004,329900,captured,1736099388,card,setl_677b1d00,2025-01-05 17:49:48
4,pay_0000000005,inv-2025-000001,264900,captured,1736952805,upi,setl_67884c00,2025-01-15 14:53:25
...,...,...,...,...,...,...,...,...
473502,pay_000007399e,INV-2026-162864,144900,captured,1784030976,upi,setl_6a56cd80,2026-07-14 12:09:36
473503,pay_000007399f,inv-2026-162865,227000,captured,1783025197,upi,setl_6a46fb80,2026-07-02 20:46:37
473504,pay_00000739a0,INV-2026-162868,12000,captured,1783009758,upi,setl_6a46fb80,2026-07-02 16:29:18
473505,pay_00000739a2,INV-2026-162873,215000,captured,1782896776,upi,setl_6a45aa00,2026-07-01 09:06:16


In [16]:
gateway.head()

,payment_id,notes_invoice_ref,amount_paise,status,created_at_epoch,method,settlement_id,created_date
0,pay_0000000002,INV-2024-000002,107000,captured,1735577378,upi,setl_67733400,2024-12-30 16:49:38
1,pay_0000000003,INV-2024-000003,35000,captured,1735653947,upi,setl_67748580,2024-12-31 14:05:47
2,pay_0000000001,INV-2024-000001,28000,captured,1735996468,upi,setl_6779cb80,2025-01-04 13:14:28
3,pay_0000000004,INV-2024-000004,329900,captured,1736099388,card,setl_677b1d00,2025-01-05 17:49:48
4,pay_0000000005,inv-2025-000001,264900,captured,1736952805,upi,setl_67884c00,2025-01-15 14:53:25


In [17]:
gateway.size

3788056

In [18]:
settlement.head()

,settlement_id,amount_paise,fees_paise,tax_paise,utr,created_at_epoch,status
0,setl_67733400,104475,2140,385,UTR1735603200244,1735639200,processed
1,setl_67748580,34174,700,126,UTR1735689600397,1735725600,processed
2,setl_6775d700,34647554,709700,127746,UTR1735776000973,1735812000,processed
3,setl_67772880,59806550,1225042,220508,UTR1735862400100,1735898400,processed
4,setl_67787a00,69685473,1427396,256931,UTR1735948800346,1735984800,processed


In [19]:
settlement['created_date']=pd.to_datetime(settlement['created_at_epoch'],unit='s')
settlement.to_csv(MERGED_DIR / 'merged_settlement.csv', index=False)

In [20]:
settlement

,settlement_id,amount_paise,fees_paise,tax_paise,utr,created_at_epoch,status,created_date
0,setl_67733400,104475,2140,385,UTR1735603200244,1735639200,processed,2024-12-31 10:00:00
1,setl_67748580,34174,700,126,UTR1735689600397,1735725600,processed,2025-01-01 10:00:00
2,setl_6775d700,34647554,709700,127746,UTR1735776000973,1735812000,processed,2025-01-02 10:00:00
3,setl_67772880,59806550,1225042,220508,UTR1735862400100,1735898400,processed,2025-01-03 10:00:00
4,setl_67787a00,69685473,1427396,256931,UTR1735948800346,1735984800,processed,2025-01-04 10:00:00
...,...,...,...,...,...,...,...,...
557,setl_6a518780,7313041,149796,26963,UTR1783728000664,1783764000,processed,2026-07-11 10:00:00
558,setl_6a52d900,4365680,89424,16096,UTR1783814400322,1783850400,processed,2026-07-12 10:00:00
559,setl_6a542a80,8395282,171964,30954,UTR1783900800671,1783936800,processed,2026-07-13 10:00:00
560,setl_6a557c00,6528796,133732,24072,UTR1783987200927,1784023200,processed,2026-07-14 10:00:00


In [21]:
settlement.size

4496

Knowing the merged columns

In [22]:
bookings.size

16286218

In [23]:
bookings.describe()

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,type_of_booking
count,1252786,1252786,1252786,1252786,1252786,1252786,1252786,1252786,1252786,1251496,1252786,1252786,1252786
unique,541465,361722,287732,6,32,3,4,183879,601,25,39,5,1
top,booking_id,17/06/2025 06:56,16/06/2025 07:59,bengaluru,BLR-01,clinic,app,PPUN0016908,Ananya Patel,T001,350,0,legacy
freq,21,25,28,301223,93640,780350,682201,125,2638,154343,112848,660547,1252786


In [24]:
bookings.groupby(['patient_id','slot_at']).nunique()

booking_id  created_at  city  clinic_id  \
patient_id  slot_at                                                     
PAHM0000003 13/08/2025 11:04           1           1     1          1   
PAHM0000006 08/12/2025 06:10           1           1     1          1   
            16/11/2025 12:34           1           1     1          1   
PAHM0000007 02/07/2025 12:16           1           1     1          1   
PAHM0000009 03/01/2026 07:24           1           1     1          1   
...                                  ...         ...   ...        ...   
PPUN0042946 12/06/2026 17:14           1           1     1          1   
PPUN0042947 13/06/2026 09:29           1           1     1          1   
PPUN0042948 28/06/2026 10:46           1           1     1          1   
PPUN0042949 17/06/2026 08:42           1           1     1          1   
patient_id  slot_at                    1           1     6          1   

                              channel  source  patient_name  test_code  \
patient_id  slot_at                                                      
PAHM0000003 13/08/2025 11:04        1       1             1          3   
PAHM0000006 08/12/2025 06:10        1       1             1          1   
            16/11/2025 12:34        1       1             1          3   
PAHM0000007 02/07/2025 12:16        1       1             1          1   
PAHM0000009 03/01/2026 07:24        1       1             1          2   
...                               ...     ...           ...        ...   
PPUN0042946 12/06/2026 17:14        1       1             1          3   
PPUN0042947 13/06/2026 09:29        1       1             1          3   
PPUN0042948 28/06/2026 10:46        1       1             1          2   
PPUN0042949 17/06/2026 08:42        1       1             1          3   
patient_id  slot_at                 1       1             1          1   

                              test_price  home_fee  type_of_booking  
patient_id  slot_at                                                  
PAHM0000003 13/08/2025 11:04           3         1                1  
PAHM0000006 08/12/2025 06:10           1         1                1  
            16/11/2025 12:34           3         1                1  
PAHM0000007 02/07/2025 12:16           1         1                1  
PAHM0000009 03/01/2026 07:24           2         1                1  
...                                  ...       ...              ...  
PPUN0042946 12/06/2026 17:14           3         1                1  
PPUN0042947 13/06/2026 09:29           3         1                1  
PPUN0042948 28/06/2026 10:46           2         1                1  
PPUN0042949 17/06/2026 08:42           3         1                1  
patient_id  slot_at                    1         1                1  

[541461 rows x 11 columns]

In [25]:
bookings.info()

<class 'pandas.DataFrame'>
RangeIndex: 1252786 entries, 0 to 1252785
Data columns (total 13 columns):
 #   Column           Non-Null Count    Dtype 
---  ------           --------------    ----- 
 0   booking_id       1252786 non-null  str   
 1   created_at       1252786 non-null  str   
 2   slot_at          1252786 non-null  str   
 3   city             1252786 non-null  str   
 4   clinic_id        1252786 non-null  str   
 5   channel          1252786 non-null  str   
 6   source           1252786 non-null  str   
 7   patient_id       1252786 non-null  str   
 8   patient_name     1252786 non-null  str   
 9   test_code        1251496 non-null  str   
 10  test_price       1252786 non-null  object
 11  home_fee         1252786 non-null  object
 12  type_of_booking  1252786 non-null  str   
dtypes: object(2), str(11)
memory usage: 124.3+ MB


In [26]:
bookings.head(10)

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,type_of_booking
0,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,legacy
1,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,legacy
2,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,legacy
3,BK-AHM-202501-000002,03/01/2025 05:18,06/01/2025 13:22,ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,legacy
4,BK-AHM-202501-000003,27/01/2025 10:46,28/01/2025 13:37,ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,legacy
5,BK-AHM-202501-000003,27/01/2025 10:46,28/01/2025 13:37,ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T017,450,150,legacy
6,BK-AHM-202501-000003,27/01/2025 10:46,28/01/2025 13:37,ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T002,120,150,legacy
7,BK-AHM-202501-000003,27/01/2025 10:46,28/01/2025 13:37,ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T006,650,150,legacy
8,BK-AHM-202501-000004,20/01/2025 02:48,24/01/2025 07:15,ahmedabad,AHM-03,clinic,app,PAHM0007069,Arjun Bose,T016,160,0,legacy
9,BK-AHM-202501-000005,06/01/2025 08:21,11/01/2025 09:56,ahmedabad,AHM-03,clinic,app,PAHM0002520,Ritika Nair,T008,1100,0,legacy


In [27]:
invoices.groupby(['booking_id','patient_id']).nunique()

,,invoice_id,invoice_date,city,channel,amount
booking_id,patient_id,,,,,
BK-AHM-202501-000001,PAHM0028525,1,1,1,1,1
BK-AHM-202501-000002,PAHM0031384,1,1,1,1,1
BK-AHM-202501-000003,PAHM0027001,1,1,1,1,1
BK-AHM-202501-000005,PAHM0002520,1,1,1,1,1
BK-AHM-202501-000006,PAHM0018893,1,1,1,1,1
...,...,...,...,...,...,...
nova_jai_0055577,PJAI0029376,1,1,1,1,1
nova_jai_0055578,PJAI0029377,1,1,1,1,1
nova_jai_0055579,PJAI0029378,1,1,1,1,1


In [28]:
invoices.describe()

,invoice_id,invoice_date,booking_id,patient_id,city,channel,amount
count,470776,470776,470776,470776,470776,470776,470776
unique,470776,548,470776,173347,6,2,1344
top,INV-2024-000001,2026-02-03,BK-BLR-202501-006763,PPUN0016908,Bengaluru,clinic,800.00
freq,1,1122,1,57,108317,284248,13956


In [29]:
invoices.info()

<class 'pandas.DataFrame'>
RangeIndex: 470776 entries, 0 to 470775
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   invoice_id    470776 non-null  str  
 1   invoice_date  470776 non-null  str  
 2   booking_id    470776 non-null  str  
 3   patient_id    470776 non-null  str  
 4   city          470776 non-null  str  
 5   channel       470776 non-null  str  
 6   amount        470776 non-null  str  
dtypes: str(7)
memory usage: 25.1 MB


In [30]:
invoices.head()

,invoice_id,invoice_date,booking_id,patient_id,city,channel,amount
0,INV-2024-000001,2024-12-30,BK-BLR-202501-006763,PBLR0044005,Bengaluru,clinic,280.00
1,INV-2024-000002,2024-12-30,BK-BLR-202501-007103,PBLR0044345,Bengaluru,clinic,"1,070.00"
2,INV-2024-000003,2024-12-31,BK-PUN-202501-001985,PPUN0008860,Pune,clinic,350.00
3,INV-2024-000004,2024-12-31,BK-CHE-202501-001176,PCHE0012203,Chennai,home,"3,299.00"
4,INV-2025-000001,2025-01-01,BK-AHM-202501-005119,PAHM0034226,Ahmedabad,clinic,"2,649.00"


In [31]:
invoices[invoices['invoice_id']=='INV-2025-000001']

,invoice_id,invoice_date,booking_id,patient_id,city,channel,amount
4,INV-2025-000001,2025-01-01,BK-AHM-202501-005119,PAHM0034226,Ahmedabad,clinic,"2,649.00"


In [32]:
gateway.describe()

,amount_paise,created_at_epoch,created_date
count,473507.000000,4.735070e+05,473507
mean,156404.737839,1.759508e+09,2025-10-03 16:07:19
min,12000.000000,1.735577e+09,2024-12-30 16:49:38
25%,70000.000000,1.747505e+09,2025-05-17 18:06:57
50%,125000.000000,1.759593e+09,2025-10-04 15:51:51
75%,207000.000000,1.771443e+09,2026-02-18 19:28:04
max,989600.000000,1.784062e+09,2026-07-14 20:49:13
std,117565.045310,1.364649e+07,NaN


In [33]:
gateway.head()

,payment_id,notes_invoice_ref,amount_paise,status,created_at_epoch,method,settlement_id,created_date
0,pay_0000000002,INV-2024-000002,107000,captured,1735577378,upi,setl_67733400,2024-12-30 16:49:38
1,pay_0000000003,INV-2024-000003,35000,captured,1735653947,upi,setl_67748580,2024-12-31 14:05:47
2,pay_0000000001,INV-2024-000001,28000,captured,1735996468,upi,setl_6779cb80,2025-01-04 13:14:28
3,pay_0000000004,INV-2024-000004,329900,captured,1736099388,card,setl_677b1d00,2025-01-05 17:49:48
4,pay_0000000005,inv-2025-000001,264900,captured,1736952805,upi,setl_67884c00,2025-01-15 14:53:25


In [34]:
gateway['notes_invoice_ref']=gateway['notes_invoice_ref'].str.upper()

In [35]:
gateway

,payment_id,notes_invoice_ref,amount_paise,status,created_at_epoch,method,settlement_id,created_date
0,pay_0000000002,INV-2024-000002,107000,captured,1735577378,upi,setl_67733400,2024-12-30 16:49:38
1,pay_0000000003,INV-2024-000003,35000,captured,1735653947,upi,setl_67748580,2024-12-31 14:05:47
2,pay_0000000001,INV-2024-000001,28000,captured,1735996468,upi,setl_6779cb80,2025-01-04 13:14:28
3,pay_0000000004,INV-2024-000004,329900,captured,1736099388,card,setl_677b1d00,2025-01-05 17:49:48
4,pay_0000000005,INV-2025-000001,264900,captured,1736952805,upi,setl_67884c00,2025-01-15 14:53:25
...,...,...,...,...,...,...,...,...
473502,pay_000007399e,INV-2026-162864,144900,captured,1784030976,upi,setl_6a56cd80,2026-07-14 12:09:36
473503,pay_000007399f,INV-2026-162865,227000,captured,1783025197,upi,setl_6a46fb80,2026-07-02 20:46:37
473504,pay_00000739a0,INV-2026-162868,12000,captured,1783009758,upi,setl_6a46fb80,2026-07-02 16:29:18
473505,pay_00000739a2,INV-2026-162873,215000,captured,1782896776,upi,setl_6a45aa00,2026-07-01 09:06:16


In [36]:
gateway.info()

<class 'pandas.DataFrame'>
RangeIndex: 473507 entries, 0 to 473506
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype        
---  ------             --------------   -----        
 0   payment_id         473507 non-null  str          
 1   notes_invoice_ref  473507 non-null  str          
 2   amount_paise       473507 non-null  int64        
 3   status             473507 non-null  str          
 4   created_at_epoch   473507 non-null  int64        
 5   method             473507 non-null  str          
 6   settlement_id      455469 non-null  str          
 7   created_date       473507 non-null  datetime64[s]
dtypes: datetime64[s](1), int64(2), str(5)
memory usage: 28.9 MB


In [37]:
gateway['settlement_id'].isna().sum()

np.int64(18038)

In [38]:
gateway_set = set(gateway['settlement_id'].dropna())
settlement_set = set(settlement['settlement_id'].dropna())

In [39]:
common_ids = gateway_set.intersection(settlement_set)

print(f"Unique IDs in Gateway: {len(gateway_set)}")
print(f"Unique IDs in Settlement: {len(settlement_set)}")
print(f"IDs that match perfectly: {len(common_ids)}")

Unique IDs in Gateway: 562
Unique IDs in Settlement: 562
IDs that match perfectly: 562


In [40]:
settlement.describe()

,amount_paise,fees_paise,tax_paise,created_at_epoch,created_date
count,5.620000e+02,5.620000e+02,562.000000,5.620000e+02,562
mean,1.237855e+08,2.535549e+06,456398.745552,1.759874e+09,2025-10-07 22:00:00
min,3.417400e+04,7.000000e+02,126.000000,1.735639e+09,2024-12-31 10:00:00
25%,1.184143e+08,2.425529e+06,436595.250000,1.747757e+09,2025-05-20 16:00:00
50%,1.251377e+08,2.563246e+06,461384.000000,1.759874e+09,2025-10-07 22:00:00
75%,1.350563e+08,2.766413e+06,497954.500000,1.771992e+09,2026-02-25 04:00:00
max,1.717846e+08,3.518734e+06,633372.000000,1.784110e+09,2026-07-15 10:00:00
std,2.313534e+07,4.738907e+05,85300.342417,1.402961e+07,NaN


In [41]:
settlement.info()

<class 'pandas.DataFrame'>
RangeIndex: 562 entries, 0 to 561
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype        
---  ------            --------------  -----        
 0   settlement_id     562 non-null    str          
 1   amount_paise      562 non-null    int64        
 2   fees_paise        562 non-null    int64        
 3   tax_paise         562 non-null    int64        
 4   utr               562 non-null    str          
 5   created_at_epoch  562 non-null    int64        
 6   status            562 non-null    str          
 7   created_date      562 non-null    datetime64[s]
dtypes: datetime64[s](1), int64(4), str(3)
memory usage: 35.3 KB


In [42]:
settlement.head(10)

,settlement_id,amount_paise,fees_paise,tax_paise,utr,created_at_epoch,status,created_date
0,setl_67733400,104475,2140,385,UTR1735603200244,1735639200,processed,2024-12-31 10:00:00
1,setl_67748580,34174,700,126,UTR1735689600397,1735725600,processed,2025-01-01 10:00:00
2,setl_6775d700,34647554,709700,127746,UTR1735776000973,1735812000,processed,2025-01-02 10:00:00
3,setl_67772880,59806550,1225042,220508,UTR1735862400100,1735898400,processed,2025-01-03 10:00:00
4,setl_67787a00,69685473,1427396,256931,UTR1735948800346,1735984800,processed,2025-01-04 10:00:00
5,setl_6779cb80,86591936,1773698,319266,UTR1736035200503,1736071200,processed,2025-01-05 10:00:00
6,setl_677b1d00,87021357,1782494,320849,UTR1736121600436,1736157600,processed,2025-01-06 10:00:00
7,setl_677c6e80,100215353,2052752,369495,UTR1736208000202,1736244000,processed,2025-01-07 10:00:00
8,setl_677dc000,105779563,2166726,390011,UTR1736294400218,1736330400,processed,2025-01-08 10:00:00
9,setl_677f1180,103589205,2121860,381935,UTR1736380800428,1736416800,processed,2025-01-09 10:00:00


In [43]:
collections.describe()

,booking_id,clinic_id,channel,status,collected_at,phlebotomist_id
count,552492,552492,552492,552492,473320,208554
unique,549598,31,2,2,283020,354
top,BK-AHM-202501-000032,BLR-01,clinic,collected,2025-02-05 07:58:00,PHBLR009
freq,2,39388,343938,473320,9,869


In [44]:
collections.groupby(['booking_id']).nunique()

,clinic_id,channel,status,collected_at,phlebotomist_id
booking_id,,,,,
BK-AHM-202501-000001,1,1,1,1,0
BK-AHM-202501-000002,1,1,1,1,0
BK-AHM-202501-000003,1,1,1,1,1
BK-AHM-202501-000005,1,1,1,1,0
BK-AHM-202501-000006,1,1,1,1,1
...,...,...,...,...,...
nova_jai_0055577,1,1,1,1,0
nova_jai_0055578,1,1,1,1,1
nova_jai_0055579,1,1,1,1,0


In [45]:
collections.info()

<class 'pandas.DataFrame'>
RangeIndex: 552492 entries, 0 to 552491
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype
---  ------           --------------   -----
 0   booking_id       552492 non-null  str  
 1   clinic_id        552492 non-null  str  
 2   channel          552492 non-null  str  
 3   status           552492 non-null  str  
 4   collected_at     473320 non-null  str  
 5   phlebotomist_id  208554 non-null  str  
dtypes: str(6)
memory usage: 25.3 MB


In [46]:
collections.head(10)

,booking_id,clinic_id,channel,status,collected_at,phlebotomist_id
0,BK-AHM-202501-000001,AHM-03,clinic,collected,2025-01-09 09:09:00,NaN
1,BK-AHM-202501-000002,AHM-04,clinic,collected,2025-01-06 13:44:00,NaN
2,BK-AHM-202501-000003,AHM-01,home,collected,2025-01-28 13:37:00,PHAHM047
3,BK-AHM-202501-000005,AHM-03,clinic,collected,2025-01-11 09:59:00,NaN
4,BK-AHM-202501-000006,AHM-04,home,collected,2025-01-11 12:39:00,PHAHM011
5,BK-AHM-202501-000007,AHM-05,home,collected,2025-01-21 11:54:00,PHAHM008
6,BK-AHM-202501-000008,AHM-01,clinic,collected,2025-01-21 08:49:00,NaN
7,BK-AHM-202501-000009,AHM-03,clinic,collected,2025-01-01 13:02:00,NaN
8,BK-AHM-202501-000010,AHM-03,home,collected,2025-01-16 17:36:00,PHAHM001
9,BK-AHM-202501-000011,AHM-01,clinic,collected,2025-01-12 10:11:00,NaN


In [47]:
merged_campaigns.describe()

,waiver_rs
count,48869.0
mean,150.0
std,0.0
min,150.0
25%,150.0
50%,150.0
75%,150.0
max,150.0


In [48]:
merged_campaigns['booking_id'].nunique()

48869

In [49]:
merged_campaigns.head(10)

,booking_id,campaign_id,waiver_rs
0,BK-AHM-202601-065286,HC-FREE-2026Q1,150
1,BK-AHM-202601-065292,HC-FREE-2026Q1,150
2,BK-AHM-202601-065298,HC-FREE-2026Q1,150
3,BK-AHM-202601-065301,HC-FREE-2026Q1,150
4,BK-AHM-202601-065306,HC-FREE-2026Q1,150
5,BK-AHM-202601-065308,HC-FREE-2026Q1,150
6,BK-AHM-202601-065310,HC-FREE-2026Q1,150
7,BK-AHM-202601-065311,HC-FREE-2026Q1,150
8,BK-AHM-202601-065317,HC-FREE-2026Q1,150
9,BK-AHM-202601-065318,HC-FREE-2026Q1,150


In [50]:
merged_campaigns.info()

<class 'pandas.DataFrame'>
RangeIndex: 48869 entries, 0 to 48868
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   booking_id   48869 non-null  str  
 1   campaign_id  48869 non-null  str  
 2   waiver_rs    48869 non-null  int64
dtypes: int64(1), str(2)
memory usage: 1.1 MB


In [51]:
patient.head(10)

,patient_id,patient_name,phone,dob,city,signup_date
0,PAHM0030088,Rahul Fernandes,+91 85778 98441,1988-02-02,Ahmedabad,2023-01-14
1,PAHM0043743,Ananya D'Souza,+91 84824 41827,1992-11-13,Ahmedabad,2025-10-02
2,PAHM0011366,Rohan Mehta,+91 77327 81379,1944-07-05,Ahmedabad,2023-01-18
3,PAHM0009021,Vihaan Pillai,+91 76067 53849,1958-03-06,Ahmedabad,2022-08-06
4,PAHM0052680,Meera Desai,+91 81849 12697,2005-11-01,Ahmedabad,2026-05-19
5,PAHM0040142,Manish Nair,+91 77339 15066,1993-05-21,Ahmedabad,2025-06-26
6,PAHM0031576,Aarav Pillai,+91 97588 60210,1990-06-15,Ahmedabad,2024-12-09
7,PAHM0031084,Meera Joshi,+91 95852 34405,2001-09-06,Ahmedabad,2023-08-14
8,PAHM0046376,Mary D'Souza,+91 74720 45867,1996-10-01,Ahmedabad,2025-12-12
9,PAHM0015250,Karan Gupta,+91 94632 45408,1997-10-06,Ahmedabad,2023-06-19


In [52]:
patient.describe()

,patient_id,patient_name,phone,dob,city,signup_date
count,309217,309217,309217,309217,309217,309217
unique,309217,1200,309200,24456,6,1403
top,PAHM0030088,Rahul Fernandes,+91 90601 56268,1900-01-01,Bengaluru,2026-01-19
freq,1,568,2,1492,71547,284


In [53]:
patient.info()

<class 'pandas.DataFrame'>
RangeIndex: 309217 entries, 0 to 309216
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   patient_id    309217 non-null  str  
 1   patient_name  309217 non-null  str  
 2   phone         309217 non-null  str  
 3   dob           309217 non-null  str  
 4   city          309217 non-null  str  
 5   signup_date   309217 non-null  str  
dtypes: str(6)
memory usage: 14.2 MB


In [54]:
results.head()

,booking_id,test_code,result_ts,result_value,result_flag,city
0,BK-AHM-202501-000001,T009,2025-01-09 20:09:00,46.8,normal,ahmedabad
1,BK-AHM-202501-000001,T004,2025-01-10 00:09:00,49.0,normal,ahmedabad
2,BK-AHM-202501-000001,T007,2025-01-09 18:09:00,74.8,normal,ahmedabad
3,BK-AHM-202501-000002,T004,2025-01-07 15:44:00,52.7,flag,ahmedabad
4,BK-AHM-202501-000003,T003,2025-01-29 05:37:00,71.7,flag,ahmedabad


In [55]:
results.groupby(['booking_id','result_ts','result_value']).nunique()

test_code  result_flag  \
booking_id           result_ts           result_value                           
BK-AHM-202501-000001 2025-01-09 18:09:00 74.8                  1            1   
                     2025-01-09 20:09:00 46.8                  1            1   
                     2025-01-10 00:09:00 49.0                  1            1   
BK-AHM-202501-000002 2025-01-07 15:44:00 52.7                  1            1   
BK-AHM-202501-000003 2025-01-29 05:37:00 71.7                  1            1   
...                                                          ...          ...   
nova_jai_0055579     2026-06-07 10:29:00 29.6                  1            1   
nova_jai_0055580     2026-06-11 13:14:00 72.2                  1            1   
nova_jai_0055581     2026-06-22 12:50:00 33.6                  1            1   
                     2026-06-22 16:50:00 12.4                  1            1   
                     2026-06-23 06:50:00 70.8                  1            1   

                                                       city  
booking_id           result_ts           result_value        
BK-AHM-202501-000001 2025-01-09 18:09:00 74.8             1  
                     2025-01-09 20:09:00 46.8             1  
                     2025-01-10 00:09:00 49.0             1  
BK-AHM-202501-000002 2025-01-07 15:44:00 52.7             1  
BK-AHM-202501-000003 2025-01-29 05:37:00 71.7             1  
...                                                     ...  
nova_jai_0055579     2026-06-07 10:29:00 29.6             1  
nova_jai_0055580     2026-06-11 13:14:00 72.2             1  
nova_jai_0055581     2026-06-22 12:50:00 33.6             1  
                     2026-06-22 16:50:00 12.4             1  
                     2026-06-23 06:50:00 70.8             1  

[1078379 rows x 3 columns]

In [56]:
results.describe()

,booking_id,test_code,result_ts,result_value,result_flag,city
count,1078420,1075209,1078420,1078420,1078420,1078420
unique,473905,24,576003,1749,2,6
top,BK-AHM-202501-000611,T001,2025-02-17 17:15:00,5.6 mmol/L,normal,bengaluru
freq,6,132304,10,3530,916384,247957


In [57]:
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 1078420 entries, 0 to 1078419
Data columns (total 6 columns):
 #   Column        Non-Null Count    Dtype
---  ------        --------------    -----
 0   booking_id    1078420 non-null  str  
 1   test_code     1075209 non-null  str  
 2   result_ts     1078420 non-null  str  
 3   result_value  1078420 non-null  str  
 4   result_flag   1078420 non-null  str  
 5   city          1078420 non-null  str  
dtypes: str(6)
memory usage: 49.4 MB


Getting cleaned data

In [58]:
gateway['settlement_id'].isna().sum()

np.int64(18038)

In [59]:
bookings

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,type_of_booking
0,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,legacy
1,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,legacy
2,BK-AHM-202501-000001,08/01/2025 03:16,09/01/2025 08:26,ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,legacy
3,BK-AHM-202501-000002,03/01/2025 05:18,06/01/2025 13:22,ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,legacy
4,BK-AHM-202501-000003,27/01/2025 10:46,28/01/2025 13:37,ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,legacy
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1252781,BK-PUN-202606-079978,25/06/2026 04:21,28/06/2026 10:46,pune,PUN-03,home,walkin,PPUN0042948,Manish Gupta,T009,800,150,legacy
1252782,BK-PUN-202606-079978,25/06/2026 04:21,28/06/2026 10:46,pune,PUN-03,home,walkin,PPUN0042948,Manish Gupta,T008,1100,150,legacy
1252783,BK-PUN-202606-079979,13/06/2026 03:03,17/06/2026 08:42,pune,PUN-04,clinic,app,PPUN0042949,Nisha Kulkarni,T001,350,0,legacy
1252784,BK-PUN-202606-079979,13/06/2026 03:03,17/06/2026 08:42,pune,PUN-04,clinic,app,PPUN0042949,Nisha Kulkarni,T008,1100,0,legacy


In [60]:
bookings['test_price'] = pd.to_numeric(bookings['test_price'], errors='coerce')
bookings['created_at']=pd.to_datetime(bookings['created_at'],errors='coerce')
cond=(bookings['created_at']>='2025-01-01') &(bookings['created_at']<'2026-01-01')
total_test_price = bookings.loc[cond,'test_price'].sum()

In [61]:
total_test_price

np.float64(215309963.0)